- nnsight.local() context sends values immediately to user's local machine from server
- Intervention graph is executed locally on downstream nodes
- Exiting local context uploads data back to server
- @nnsight.trace function decorator enables functions to be added to intervention graph when using nnsight.local()

## nnsight.local()

You may sometimes want to locally access and manipulate values during remote execution. Using nnsight.local() on a proxy, you can send remote content to your local machine and apply local functions. The intervention graph is then executed locally on downstream nodes (until you send execution back to the remote server by exiting the .local() context.)

In [38]:
from nnsight import LanguageModel
model = LanguageModel("deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B")

In [24]:
# This will give you a remote LOG response because it's coming from the remote server
with model.trace("hello") as tracer:
    hs = model.transformer.h[-1].output[0]
    print(hs[0,0,0])
    out = model.lm_head.output.save()
print(out)

tensor(-0.0647, grad_fn=<SelectBackward0>)
tensor([[[-37.0707, -36.4855, -40.3520,  ..., -46.5168, -45.4142, -37.9090]]],
       grad_fn=<UnsafeViewBackward0>)


## @nnsight.trace function decorator

We can use function decorators to create custom functions to be used during .local calls. this is a handy way to enable live streaming of a chat or to train probing classifiers on model hidden states.

In [45]:
import nnsight
@nnsight.trace
def my_decoding_function(tokens, model, max_length=80, state=None):
    # Initialize state if not provided
    if state is None:
        state = {'current_line': '', 'current_line_length': 0}

    token = tokens[-1] # only use last token

    # Decode the token
    decoded_token = model.tokenizer.decode(token).encode("unicode_escape").decode()

    if (decoded_token == '\\n') or (decoded_token == '\n'):  # Handle explicit newline tokens
        # Print the current line and reset state
        print('',flush=True)
        state['current_line'] = ''
        state['current_line_length'] = 0
    else:
        # Check if adding the token would exceed the max length
        if state['current_line_length'] + len(decoded_token) > max_length:
            print('',flush=True)
            state['current_line'] = decoded_token  # Start a new line with the current token
            state['current_line_length'] = len(decoded_token)
            print(decoded_token, flush=True, end="")  # Print ONLY the new token
        else:
            # Add a space if the line isn't empty and append the token
            if state['current_line']:
                state['current_line'] += decoded_token
            else:
                state['current_line'] = decoded_token
            state['current_line_length'] += len(decoded_token)
            print(decoded_token, flush=True, end="")  # Print ONLY the new token

    return state

/disk/u/gio/.conda/envs/retrieval/lib/python3.11/site-packages/nnsight/__init__.py:80: UserWarning: nnsight.trace is deprecated as of v0.5.0 and will be removed in a future version.
  warnings.warn(deprecation_message)


In [47]:
import torch

nnsight.CONFIG.APP.REMOTE_LOGGING = False

prompt = "A press release is an official statement delivered to members of the news media for the purpose of"

# Initialize the state for decoding
state = {'current_line': '', 'current_line_length': 0}

all_tokens = []

with model.generate(prompt, max_new_tokens=20, do_sample=False) as tracer:

    with tracer.all():

        # Access model output
        out = model.lm_head.output.save()

        # Apply softmax to obtain probs and save the result
        probs = torch.nn.functional.softmax(out, dim=-1)
        max_probs = torch.max(probs, dim=-1)
        tokens = max_probs.indices.cpu().tolist()
        all_tokens.append(tokens[0]).save()

        state = my_decoding_function(tokens[0], model, max_length=80, state=state)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


 providing information to the public. It is a statement that is official, not a
 statement that is merely